# Réplication — *The Premium, Not the Forecast*

Ce notebook régénère **chaque chiffre du papier** et les confronte aux valeurs imprimées.
La construction principale est la **parité call-put** (`ts_reinvert.parquet`) ; la construction
par IV du fournisseur (`ts_optionsdx.parquet`) sert de robustesse (Annexe B).

**Comment le lire.** Chaque section appelle une partie de `paper_numbers.py`, qui est la
source unique de vérité. La dernière cellule compare tout au dictionnaire `PAPER`, recopié
à la main depuis le PDF : si un chiffre du code et un chiffre du papier divergent, elle dit
lequel. Il n'y a donc aucun moyen que le papier et le code s'écartent sans que ce notebook
le signale.

**Durée.** Environ dix minutes. Les prévisions ARMA sont mises en cache dans `cache/`
(elles coûtent une vingtaine de minutes à recalculer) ; la cellule de garde explique comment
les régénérer.

In [ ]:
import sys, os, json, pickle, warnings, platform
warnings.filterwarnings("ignore")
sys.path.insert(0, ".")
import numpy as np, pandas as pd
import vix_lab as vl, ts_extract as tx, futures_signed as fs, ts_models as tm
import paper_numbers as pn

print("python %s | numpy %s | pandas %s" % (platform.python_version(), np.__version__, pd.__version__))
for f in ("ts_reinvert.parquet", "ts_optionsdx.parquet",
          "cache/preds_reinvert.pkl", "cache/arma_bic_reinvert.pkl"):
    print(("  OK   " if os.path.exists(f) else "  MANQUE ") + f)

## 0. Garde-fou : alignement des prévisions

Le risque numéro un dans une évaluation walk-forward est le décalage d'un jour entre la date
d'entraînement et la date notée. Il ne provoque aucune erreur, seulement des chiffres faux.

Trois vérifications. L'erreur-type de Lo (2002) validée contre sa forme fermée sous i.i.d. ;
puis, sur une **marche aléatoire simulée**, la démonstration que décaler une prévision d'un
jour produit exactement $1-\sqrt{1+1/h}$, soit $-41\%$ à un jour et $-9{,}5\%$ à cinq ; puis,
sur la vraie série, la vérification qu'aucun modèle ne gagne anormalement à l'horizon d'un
jour, ce qui trahirait une fuite d'information.

La deuxième est la plus importante : elle teste le détecteur lui-même. Sur la série réelle,
qui revient à la moyenne, le même décalage ne coûterait que $-18\%$ à un jour et presque rien
à trente, donc c'est bien sur une marche aléatoire pure qu'il faut l'étalonner.

In [ ]:
# --- (a) l'erreur-type de Lo contre sa forme fermee sous i.i.d.
rng = np.random.default_rng(0); r = rng.normal(0.0004, 0.01, 4000)
sr, se, T, _ = fs.sharpe_se(r); s = sr/np.sqrt(252)
ref = np.sqrt(252*(1+s**2/2)/T)
print("SE de Lo : estimee %.4f | analytique %.4f" % (se, ref))
assert abs(se-ref) < 0.01, "estimateur d'erreur-type invalide"

# --- (b) le detecteur de decalage, sur une marche aleatoire PURE
# Sur une marche aleatoire, prevoir la valeur courante donne un skill exactement nul, et
# decaler la prevision d'un jour donne 1 - sqrt(1 + 1/h) : -41% a un jour, -9.5% a cinq.
# Si cette cellule cessait d'afficher ces valeurs, le detecteur serait casse.
idx = pd.bdate_range("2000-01-03", periods=3000)
y = pd.Series(np.cumsum(rng.normal(0, 1, 3000)) + 20, index=idx)
for h in (1, 5):
    aligne = 100*vl.skill(y.iloc[1000:], y, h)
    decale = 100*vl.skill(y.shift(1).iloc[1000:].dropna(), y, h)
    theorie = 100*(1 - np.sqrt(1 + 1/h))
    print("  RW simulee h=%d : aligne %+.2f%% | decale %+.1f%% (theorie %+.1f%%)" % (h, aligne, decale, theorie))
    assert abs(aligne) < 0.5,            "la prevision alignee devrait donner un skill nul"
    assert abs(decale - theorie) < 1.5,  "le detecteur de decalage ne repond plus a la theorie"

# --- (c) sur la vraie serie, aucun forecaster ne doit gagner a un jour
P = vl.load_pipeline("ts_reinvert.parquet")
iv30, si = P["iv30"], P["si"]
pre = pickle.load(open("cache/preds_reinvert.pkl", "rb"))
arma = pickle.load(open("cache/arma_bic_reinvert.pkl", "rb"))
pre[1]["ARMA"] = arma[("iv30", 1)]
for m in ("ARMA", "HAR", "VAR", "Ridge"):
    sk = 100*vl.skill(pre[1][m], iv30, 1)
    print("  %-6s h=1 : skill %+.1f%%" % (m, sk))
    assert -2 < sk < 4, "skill anormal a un jour : suspicion de fuite d'information"

print("\nGARDE-FOU OK")

## 1. La courbe et ses trois facteurs (§2, Tableau 1)

In [ ]:
pn.part_stat()

## 2. Prévision : Tableaux 2, 3 et 5

`part_stat` ci-dessus a déjà produit les skills (Tableau 2), les benchmarks ancrés
(Tableau 3), la décomposition par facteur (Tableau 5) et les contributions en points de
volatilité. On passe aux trois contrôles de la §3.3.

In [ ]:
pn.part_mcs()      # ensemble de confiance : bloc max(2h,5) contre le bloc conventionnel

In [ ]:
pn.part_cw()       # Clark-West + sensibilite a la troncature de Newey-West
pn.part_boot()     # intervalle bootstrap du skill

In [ ]:
pn.part_mult()     # famille de 42 tests : Holm et Benjamini-Hochberg

### Le skill de la courbure vient-il d'une poignée de jours ?

C'est le reproche naturel face à un skill de 20 à 26 % sur $\beta_2$. La réponse est que la
concentration existe mais qu'elle est **universelle** : une part supérieure à 100 % signifie
que les cinquante jours principaux portent plus que le gain total, donc que le reste est
collectivement négatif. C'est le cas de dix-neuf cellules sur vingt-quatre. La courbure sous
ridge est la moins concentrée du tableau, et la seule à rester positive après retrait des
cinquante jours les plus influents.

In [ ]:
pn.part_conc()
d = json.load(open("results/numbers.json"))["conc"]
tab = pd.DataFrame(d).T[["rmse","mae","days_won","top50_share","rmse_ex_top50"]]
tab.columns = ["RMSE %","MAE %","jours gagnes %","top50 du gain %","RMSE hors top50 %"]
display(tab.sort_values("top50 du gain %"))

## 3. Le signal est-il distinct du prix ? (§4, Tableaux 6 et 7)

In [ ]:
pn.part_fwd()      # ensemble de confiance avec le forward de la courbe en concurrent

In [ ]:
pn.part_span()     # regression de spanning en espace negocie, puis sur le rendement monetise

## 4. Le test économique (§5, Tableaux 8 et 9)

In [ ]:
pn.part_econ()

### Décomposition exposition / timing

$\mathbb{E}[w\,r] = \mathbb{E}[w]\,\mathbb{E}[r] + \mathrm{Cov}(w,r)$ sépare ce qu'une règle
gagne par son exposition moyenne de ce qu'elle gagne en faisant varier cette exposition.
C'est ce qui montre que la règle projetée est réellement plate en moyenne, que la perte de la
règle spot est une exposition nette longue, et que l'avance du placebo tient surtout à un
livre plus court.

In [ ]:
pn.part_decomp()
d = json.load(open("results/numbers.json"))["decomp"]
display(pd.DataFrame(d).T[["total","exposure","timing","mean_w","timing_sharpe"]])

## 5. Le spread calendaire (§5.1, Tableau 10)

In [ ]:
import slope_spread as ss
tab_slope = ss.main("ts_reinvert.parquet")

## 6. Annexes : construction alternative, cible brute, régime

In [ ]:
pn.part_alt()      # Annexe B : IV du fournisseur (robustesse)
pn.part_app()      # cible brute et performance par regime
pn.part_eiv()      # erreurs sur variables : ajustement pair contre impair
pn.part_extra()    # seuil, queues, evenements, beta de la projection
pn.part_coverage() # couverture de la borne appariee


## 7. Vérification finale

`PAPER` recopie à la main les valeurs **telles qu'imprimées dans le PDF**. La cellule les
confronte à ce que le code vient de calculer. Toute divergence est affichée avec son écart.
C'est le seul mécanisme qui empêche le papier et le code de dériver l'un par rapport à l'autre.

In [ ]:
N = json.load(open("results/numbers.json"))

PAPER = {
 # ---- Tableau 1 et §2
 "n jours de facteurs":            (N["stat"]["n_days"],                       3354,  0.5),
 "R2 median du fit":               (N["stat"]["r2_med"],                       0.966, 0.001),
 # ---- Tableau 2 (skills vs marche aleatoire)
 "Tab.2 ARMA h=30":                (N["stat"]["skills"]["ARMA"][5],            13.1,  0.06),
 "Tab.2 HAR h=30":                 (N["stat"]["skills"]["HAR"][5],             12.8,  0.06),
 "Tab.2 VAR h=30":                 (N["stat"]["skills"]["VAR"][5],             13.4,  0.06),
 "Tab.2 Ridge h=30":               (N["stat"]["skills"]["Ridge"][5],           13.9,  0.06),
 "Tab.2 Ridge h=14":               (N["stat"]["skills"]["Ridge"][4],            6.7,  0.06),
 # ---- Tableau 3 (benchmarks ancres)
 "Tab.3 RMSE du RW h=30":          (N["stat"]["anchor"]["30"]["RW"],           7.677, 0.002),
 "Tab.3 RMSE du ridge h=30":       (N["stat"]["anchor"]["30"]["Ridge"],        6.620, 0.002),
 "Tab.3 skill vs AR(1) h=30":      (N["stat"]["anchor"]["30"]["s_ar"],          6.5,  0.06),
 "ecart-type de iv30 sur le test": (N["stat"]["sd_test"],                      7.54,  0.01),
 # ---- §3.3
 "MCS p d'arret h=30":             (N["mcs"]["30"]["p"],                       0.12,  0.006),
 "Clark-West ridge h=30":          (N["cw"]["iv30"]["30"]["Ridge"],            1.95,  0.006),
 "Clark-West a 7 retards":         (N["cw"]["trunc"]["7"][0],                  2.83,  0.006),
 "famille de tests":               (N["mult"]["n_tests"],                      42,    0.5),
 "rejets individuels":             (N["mult"]["n_raw"],                        37,    0.5),
 "survivants Holm":                (N["mult"]["n_holm"],                        7,    0.5),
 "survivants BH":                  (N["mult"]["n_bh"],                         37,    0.5),
 "borne basse du bootstrap h=30":  (N["boot"]["30"][1],                        3.9,   0.06),
 # ---- Tableau 5 et §3.4
 "Tab.5 Ridge beta2 h=30":         (N["stat"]["factors"]["beta2"]["Ridge"][2], 26.4,  0.06),
 "Tab.5 Ridge beta1 h=30":         (N["stat"]["factors"]["beta1"]["Ridge"][2], 19.4,  0.06),
 "contribution de la pente 30j":   (N["stat"]["contrib30"][1],                 5.08,  0.006),
 "contribution courbure 30j":      (N["stat"]["contrib30"][2],                 1.04,  0.006),
 "pic du chargement courbure":     (N["stat"]["curv_peak"][0],                 0.2984,0.0001),
 "courbure : MAE h=14":            (N["conc"]["Ridge/beta2/h=14"]["mae"],      16.9,  0.06),
 "courbure : jours gagnes h=14":   (N["conc"]["Ridge/beta2/h=14"]["days_won"], 56.3,  0.06),
 "courbure : top50 h=14":          (N["conc"]["Ridge/beta2/h=14"]["top50_share"], 78,  0.5),
 "courbure : hors top50 h=30":     (N["conc"]["Ridge/beta2/h=30"]["rmse_ex_top50"], 9.2, 0.06),
 # ---- Tableau 6
 "Tab.6 RMSE forward h=5":         (N["fwd"]["table"]["5"]["Forward"],         3.602, 0.002),
 "Tab.6 RMSE ridge h=30":          (N["fwd"]["table"]["30"]["Ridge"],          6.620, 0.002),
 # ---- Tableau 7
 "Tab.7 t du modele h=30":         (N["span"]["price"]["30"][1],               4.48,  0.006),
 "Tab.7 t de la base h=30":        (N["span"]["price"]["30"][3],               1.57,  0.006),
 "Tab.7 R2 h=30":                  (N["span"]["price"]["30"][4],               0.178, 0.0006),
 "Tab.7 R2 sur le rendement h=30": (N["span"]["return"]["30"][1],              0.020, 0.0006),
 # ---- §5.2 : instrument et base
 "base : intercept":               (N["econ"]["basis"]["a"],                   7.26,  0.006),
 "base : pente":                   (N["econ"]["basis"]["b"],                   0.809, 0.0006),
 "base : R2":                      (N["econ"]["basis"]["r2"],                  0.884, 0.0006),
 "base moyenne F - iv30":          (N["econ"]["basis"]["mean"],                4.27,  0.006),
 # ---- Tableau 8
 "Tab.8 modele h=30":              (N["econ"]["projected"]["30"]["model"],     0.29,  0.006),
 "Tab.8 placebo h=30":             (N["econ"]["projected"]["30"]["placebo"],   0.46,  0.006),
 "Tab.8 difference h=30":          (N["econ"]["projected"]["30"]["diff"],     -0.17,  0.006),
 "Tab.8 t de la difference h=30":  (N["econ"]["projected"]["30"]["t"],        -0.61,  0.006),
 "Tab.8 modele h=5":               (N["econ"]["projected"]["5"]["model"],      0.35,  0.006),
 "Tab.8 passif":                   (N["econ"]["passive"]["sharpe"],            0.30,  0.006),
 "Tab.8 spot h=5":                 (N["econ"]["spot"]["5"],                   -0.43,  0.006),
 "long % h=30":                    (N["econ"]["projected"]["30"]["long"],      33,    0.5),
 "positions opposees % h=30":      (N["econ"]["projected"]["30"]["opp_pct"],   22,    0.5),
 "roll : unites par an":           (N["econ"]["roll"]["per_year"],             25.1,  0.06),
 # ---- Tableau 9
 "Tab.9 decale d'un jour h=30":    (N["econ"]["robust"]["lagged"]["30"][2],   -0.24,  0.006),
 "Tab.9 plein horizon h=30":       (N["econ"]["robust"]["fullhorizon"]["30"][2], -0.07, 0.006),
 # ---- decomposition
 "decomp projetee : exposition":   (N["decomp"]["projected/h=30"]["exposure"],  0.1,  0.06),
 "decomp projetee : timing":       (N["decomp"]["projected/h=30"]["timing"],    9.7,  0.06),
 "decomp spot : exposition":       (N["decomp"]["spot/h=30"]["exposure"],      -4.3,  0.06),
 "decomp placebo : exposition":    (N["decomp"]["placebo/h=30"]["exposure"],    3.8,  0.06),
 "decomp projetee : total":        (N["decomp"]["projected/h=30"]["total"],     9.8,  0.06),
 "decomp signe a echelle moyenne": (N["decomp"]["projected/h=30"]["sign_at_mean_scale"], 10.3, 0.06),
 "decomp interaction":             (N["decomp"]["projected/h=30"]["interaction"], -0.5,  0.06),
 # ---- Annexe B
 "Annexe B : construction, corr":  (N["alt"]["corr"],                          0.997, 0.0006),
 "Annexe B : ecart moyen (pt)":    (N["alt"]["mad"],                           0.452, 0.0006),
 "Annexe B : modele h=30":         (N["alt"]["projected"]["30"][0],            0.36,  0.006),
 "Annexe B : placebo h=30":        (N["alt"]["projected"]["30"][1],            0.37,  0.006),
 # ---- Annexe A
 "Annexe A : cible brute h=30":    (N["app"]["raw"]["skill_fit_vs_raw"]["30"][1], 13.2, 0.06),
 "Annexe A : regime calme h=30":   (N["app"]["regime"]["30"]["calm"][0],       1.22,  0.006),
 "Annexe A : regime volatil h=30": (N["app"]["regime"]["30"]["vol"][0],       -0.67,  0.006),
 # ---- couverture de la borne appariee (la contribution du papier)
 "couverture bloc 21":             (N["coverage"]["blocks"]["21"]["coverage"], 95.6,  0.6),
 "couverture bloc 63":             (N["coverage"]["blocks"]["63"]["coverage"], 96.7,  0.6),
 "SE HAC simule":                  (N["coverage"]["blocks"]["21"]["se_hac"],   0.302, 0.006),
 "kurtosis du modele":             (N["coverage"]["kurtosis"][0],              281,   1.0),
 # ---- erreurs sur variables
 "fiabilite beta2":                (N["eiv"]["beta2"]["reliability"],          0.828, 0.002),
 "fiabilite beta0":                (N["eiv"]["beta0"]["reliability"],          0.970, 0.002),
 "skill mecanique beta2 h=14":     (N["eiv"]["beta2"]["mechanical_h14"],       4.7,   0.06),
 "skill mecanique beta2 h=30":     (N["eiv"]["beta2"]["mechanical_h30"],       4.1,   0.06),
 # ---- projection : niveau contre variations
 "beta de la projection, niveau":  (N["extra"]["beta_level_vs_change"]["level"], 0.809, 0.002),
 "beta sur variations h=30":       (N["extra"]["beta_level_vs_change"]["h30"],   0.654, 0.006),
 "beta sur variations h=1":        (N["extra"]["beta_level_vs_change"]["h1"],    0.491, 0.006),
 # ---- seuil, queues, evenements
 "seuil delta=1.2 h=30":           (N["extra"]["delta"]["30"]["1.2"][0],       -0.20,  0.006),
 "volatilite du modele":           (N["extra"]["vol"]["model"],                22.7,   0.06),
 "volatilite du placebo":          (N["extra"]["vol"]["placebo"],              22.3,   0.06),
 "difference hors fev 2018":       (N["extra"]["events"]["ex_feb2018"][0],     -0.23,  0.006),
 "difference hors mars 2020":      (N["extra"]["events"]["ex_mar2020"][0],     -0.10,  0.006),
}

echecs = []
for nom, (calcule, papier, tol) in PAPER.items():
    ok = abs(float(calcule) - float(papier)) <= tol
    if not ok:
        echecs.append((nom, calcule, papier))
    print("  %s %-34s calcule %10.4f   papier %10.4f" %
          ("OK  " if ok else "ECHEC", nom, float(calcule), float(papier)))

print("\n%d verifications, %d echec(s)" % (len(PAPER), len(echecs)))
if echecs:
    print("\nDivergences papier / code :")
    for nom, c, p in echecs:
        print("   %-34s code %.4f  vs  papier %.4f" % (nom, c, p))
    raise AssertionError("le papier et le code ont diverge : corriger l'un ou l'autre")
print("Le papier et le code sont coherents.")